# HW1: RAW Image Processing

In [2]:
# I did this in colab, so I just copied over all the requirements.txt
!pip install numpy matplotlib imageio scikit-image>=0.25 scikit-learn ipywidgets ipykernel rawpy

In [3]:
# All imports
import rawpy
import matplotlib as plt
import numpy as np
from scipy.ndimage import correlate
from imageio import imwrite

In [4]:
# Constants

# filename = 'L1004220'
# filename = 'L1004235'
filename = 'L1004432'

path_in = f'raw/{filename}.DNG'
path_out = f'processed/{filename}.jpg'
expected_path = f'example-processed/{filename}.jpg'

In [16]:
# 1) Convert the raw image data to 32-bit floating point in [0, 1] range
def convert_to_floating_pt(path):
  # Open the raw image
  with rawpy.imread(path) as raw:
    raw_image = raw.raw_image.copy().astype(dtype=np.float32)

    # Find the max value in the raw image
    max_val = np.max(raw_image)

    # Divide all values in the image by the max value, ensuring nothing exceeds 1
    return raw_image/max_val

In [6]:
# 2) Create the filter kernels you will need for the Bayer demosaicing

# Average of horizontal and vertical neighbors
# (Use when on Blue or Red, trying to get Green)
G_FILTER = np.array([[0, 0.25, 0], [0.25, 0, 0.25], [0, 0.25, 0]])

# Average of horizontal neighbors
# (Use on Green trying to get Red or Blue)
RB_H_FILTER = np.array([[0, 0, 0], [0.5, 0, 0.5], [0, 0, 0]])

# Average of vertical neighbors
# (Use on Green trying to get Red or Blue)
RB_V_FILTER = np.array([[0, 0.5, 0], [0, 0, 0], [0, 0.5, 0]])

# Average of diagonal neighbors
# (Use on Red or Blue to get Blue or Red); (ex, if on blue, use to get red)
RB_D_FILTER = np.array([[0.25, 0, 0.25], [0, 0, 0], [0.25, 0, 0.25]])


In [7]:
# 3/4/5) Apply Filters
def apply_red_filter(floating_img):
  # Everything but the red tiles will be zeroed out
  red = np.zeros_like(floating_img)
  red[::2, ::2] = floating_img[::2, ::2]

  # Apply the horizontal red filter
  red[::2, 1::2] = correlate(red, RB_H_FILTER, mode='mirror')[::2, 1::2]

  # Apply the vertical red filter
  red[1::2, ::2] = correlate(red, RB_V_FILTER, mode='mirror')[1::2, ::2]

  # Apply the diagonal red filter
  red[1::2, 1::2] = correlate(red, RB_D_FILTER, mode='mirror')[1::2, 1::2]

  return red

def apply_green_filter(floating_img):
  green = np.zeros_like(floating_img)
  green[1::2, ::2] = floating_img[1::2, ::2]
  green[::2, 1::2] = floating_img[::2, 1::2]

  # Apply the only green filter (for blue and red squared)
  green[::2, ::2] = correlate(green, G_FILTER, mode='mirror')[::2, ::2]
  green[1::2, 1::2] = correlate(green, G_FILTER, mode='mirror')[1::2, 1::2]

  return green

def apply_blue_filter(floating_img):
  blue = np.zeros_like(floating_img)
  blue[1::2, 1::2] = floating_img[1::2, 1::2]

  # Apply the horizontal blue filter
  blue[1::2, ::2] = correlate(blue, RB_H_FILTER, mode='mirror')[1::2, ::2]

  # Apply the vertical blue filter
  blue[::2, 1::2] = correlate(blue, RB_V_FILTER, mode='mirror')[::2, 1::2]

  # Apply the diagonal blue filter
  blue[::2, ::2] = correlate(blue, RB_D_FILTER, mode='mirror')[::2, ::2]

  return blue

In [8]:
# 6) Stack all r, g, b channels into one (still floating point)
def stack_interpolated_channels(r, g, b):
  return np.stack((r, g, b), axis=-1)

In [9]:
# 7) White Balance
def apply_white_balance(image, t_mean=0.25):
  balanced = image.copy()

  r_coeff = t_mean/(np.mean(balanced[:, :, 0]))
  g_coeff = t_mean/(np.mean(balanced[:, :, 1]))
  b_coeff = t_mean/(np.mean(balanced[:, :, 2]))

  balanced[:, :, 0] *= r_coeff
  balanced[:, :, 1] *= g_coeff
  balanced[:, :, 2] *= b_coeff

  return balanced


In [10]:
# 8) Apply an Inverse Gamma curve
def apply_inverse_gamma_curve(image, inv_gamma=0.55):
  # For every pixel, raise it to the power of (1/gamma)
  return image ** inv_gamma

In [11]:
# 9) Clip and Quantize
def clip(image):
  # CLip to [0, 1]
  return np.clip(image, 0.0, 1.0)

def scale(image):
  # Scale it to 255
  return image * 255.0

def to_8_bit(image):
  # Round it to the nearest int, and then convert it to uint8
  return np.round(image).astype(np.uint8)

In [12]:
# 10) Save the image to Disk
def save_image(image):
  imwrite(path_out, image)

In [17]:
# Main Function
def main():
  # Demosaic
  floating_img = convert_to_floating_pt(path_in)

  red_channel = apply_red_filter(floating_img)
  green_channel = apply_green_filter(floating_img)
  blue_channel = apply_blue_filter(floating_img)

  image = stack_interpolated_channels(red_channel, green_channel, blue_channel)

  # White Balance
  image = apply_white_balance(image)

  # Gamma Curve
  image = apply_inverse_gamma_curve(image)

  # Clip
  image = clip(image)

  # Scale
  image = scale(image)

  # Convert to 8-bit unsigned
  image = to_8_bit(image)

  # Write the image to disk
  save_image(image)

main()